# Oracle-based methods

These methods estimate useful feature acquisitions from the data distribution. AFABench includes AACO, which uses oracle-based lookahead.

AFABench examples: AACO. No algorithm has been implemented in this notebook yet.

Data preparation is included below. Use the AFABench Python 3.12.10 environment and the same patient counts and seed as the other category notebooks. No other notebook needs to run first.

## imports and settings

Choose the patient counts and seed, and load the evidence and diagnosis descriptions.

In [4]:
from pathlib import Path
import pandas as pd
import torch
import ast

trainNo = 2000
validNo = 500
testNo = 500
rndseed = 42


project = Path(r"C:\Users\ragha\OneDrive\Desktop\SDP-Sequential-Diagnosis")
data = project / "data"
evidences = pd.read_json(data / "release_evidences.json",orient="index")

conditions = pd.read_json( data / "release_conditions.json", orient="index")

print("Evidence types:", len(evidences))
print("Diagnoses:", len(conditions))

Evidence types: 223
Diagnoses: 49


## load patient data

Read the training, validation, and test files and select the patients for this experiment. Use the same seed and patient counts when comparing methods.

In [5]:

columns = ["PATHOLOGY", "EVIDENCES", "INITIAL_EVIDENCE"]

train_df = pd.read_csv(
    data / "release_train_patients.zip",
    usecols=columns
)

validation_df = pd.read_csv(
    data / "release_validate_patients.zip",
    usecols=columns
)

test_df = pd.read_csv(
    data / "release_test_patients.zip",
    usecols=columns
)

train_df = train_df.sample(n=min(trainNo, len(train_df)), random_state=rndseed)
validation_df = validation_df.sample( n=min(validNo, len(validation_df)), random_state=rndseed)
test_df = test_df.sample( n=min(testNo, len(test_df)), random_state=rndseed)

print("Train:", len(train_df), "Validation:", len(validation_df), "Test:", len(test_df))
train_df.head(3)

Train: 2000 Validation: 500 Test: 500


,PATHOLOGY,EVIDENCES,INITIAL_EVIDENCE
625083,Allergic sinusitis,"['E_86', 'E_87', 'E_124', 'E_169', 'E_181', 'E...",E_169
469074,Myasthenia gravis,"['E_28', 'E_38', 'E_52', 'E_63', 'E_65', 'E_17...",E_38
69338,Influenza,"['E_50', 'E_53', 'E_54_@_V_161', 'E_54_@_V_183...",E_129


### define feature columns
Create a column for each symptom or possible answer.
Use the same columns across the training, validation, and test splits.

In [6]:
feature_columns = []
default_answers = {}

for row, evidence in evidences.iterrows():
    evidence_name = evidence["name"]

    if evidence["data_type"] == "B":
        feature_columns.append(evidence_name)

    else:
        for answer in evidence["possible-values"]:
            feature_columns.append(f"{evidence_name}_@_{answer}")

        default_answers[evidence_name] =(f"{evidence_name}_@_{evidence['default_value']}")

print("Number of feature columns:", len(feature_columns))

Number of feature columns: 972


### encode patient evidence

Use 1 for recorded symptoms or answers and 0 for the others.
For unrecorded nonbinary evidence, use the dataset's default answer.

In [7]:
reformat = []

for frame in [train_df, validation_df, test_df]:
    features = pd.DataFrame(
        0,
        index=frame.index,
        columns=feature_columns,
        dtype="float32"
    )

    for index, row in frame.iterrows():
        patient_evidences = ast.literal_eval(row["EVIDENCES"])
        recorded_evidence = []

        for answer in patient_evidences:
            assert answer in features.columns, f"Unknown evidence: {answer}"

            features.at[index, answer] = 1

            evidence_name = answer.split("_@_", 1)[0]
            recorded_evidence.append(evidence_name)

        # Add default answers for unrecorded nonbinary evidence.
        for evidence_name, default_answer in default_answers.items():
            if evidence_name not in recorded_evidence:
                features.at[index, default_answer] = 1

    reformat.append(features)

train_features, validation_features, test_features = reformat

print("Training:", train_features.shape)
print("Validation:", validation_features.shape)
print("Test:", test_features.shape)

Training: (2000, 972)
Validation: (500, 972)
Test: (500, 972)


### encode diagnosis labels

Convert PATHOLOGY into one-hot encoding.

In [8]:
diagnoses = sorted(conditions["cond-name-eng"].unique())

reformat = []

for patient in [train_df, validation_df, test_df]:
    assert patient["PATHOLOGY"].isin(diagnoses).all(), "Unknown diagnosis found"

    labels = pd.get_dummies(patient["PATHOLOGY"])
    labels = labels.reindex(columns=diagnoses, fill_value=0)
    labels = labels.astype("float32")

    reformat.append(labels)

train_labels, validation_labels, test_labels = reformat

print("Training labels:", train_labels.shape)
print("Validation labels:", validation_labels.shape)
print("Test labels:", test_labels.shape)

Training labels: (2000, 49)
Validation labels: (500, 49)
Test labels: (500, 49)


## convert data to PyTorch tensors

Convert the encoded tables to the numeric arrays used by the models. Inputs contain evidence values; targets contain the known diagnoses.

In [9]:
train_inputs = torch.tensor(train_features.to_numpy(), dtype=torch.float32)
train_targets = torch.tensor(train_labels.to_numpy(), dtype=torch.float32)

validation_inputs = torch.tensor(validation_features.to_numpy(), dtype=torch.float32)
validation_targets = torch.tensor(validation_labels.to_numpy(), dtype=torch.float32)

test_inputs = torch.tensor(test_features.to_numpy(), dtype=torch.float32)
test_targets = torch.tensor(test_labels.to_numpy(), dtype=torch.float32)

print("Training:", train_inputs.shape, train_targets.shape)
print("Validation:", validation_inputs.shape, validation_targets.shape)
print("Test:", test_inputs.shape, test_targets.shape)

Training: torch.Size([2000, 972]) torch.Size([2000, 49])
Validation: torch.Size([500, 972]) torch.Size([500, 49])
Test: torch.Size([500, 972]) torch.Size([500, 49])


## prepare training batches

Pair each patient's evidence with their diagnosis and group patients into batches. Models can use these batches during training and evaluation; the tensors are also available for methods that use them directly.

In [11]:
from torch.utils.data import TensorDataset, DataLoader

batch_size = 128
reformat = []

for inputs, targets in [
    (train_inputs, train_targets),
    (validation_inputs, validation_targets),
    (test_inputs, test_targets)
]:
    patients = TensorDataset(inputs, targets)
    reformat.append(patients)

train_dataset, validation_dataset, test_dataset = reformat

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    drop_last=True
)

validation_loader = DataLoader(validation_dataset, batch_size=batch_size)

test_loader = DataLoader(test_dataset,batch_size=batch_size)

## method implementation

Add the selected algorithm's setup, training or fitting, model saving, and evaluation here. Running this notebook currently prepares the data only.